In [ ]:
# Import required libraries
import spacy
import gensim
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import torch

In [ ]:

# Load SpaCy for text preprocessing
nlp = spacy.load("en_core_web_sm")

In [ ]:

# Function to preprocess text using SpaCy
def preprocess_text(text):
    doc = nlp(text.lower())
    return ' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])


In [ ]:
# Function for TF-IDF Vectorization
def tfidf_vectorizer(documents):
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(documents)
    return tfidf_matrix

In [ ]:
# Function for Word2Vec embedding
def word2vec_vectorizer(documents):
    # Tokenize and prepare the text for Word2Vec
    tokenized_documents = [doc.split() for doc in documents]
    
    # Train a Word2Vec model on the documents
    model = Word2Vec(tokenized_documents, vector_size=100, window=5, min_count=1, workers=4)
    
    # Get the Word2Vec vector for each document
    doc_vectors = []
    for doc in tokenized_documents:
        vector = sum([model.wv[word] for word in doc if word in model.wv])
        doc_vectors.append(vector / len(doc))  # average vector for document
    
    return doc_vectors

In [ ]:

# Function for BERT embedding using HuggingFace transformers
def bert_vectorizer(documents):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')

    doc_vectors = []
    for doc in documents:
        inputs = tokenizer(doc, return_tensors='pt', padding=True, truncation=True, max_length=512)
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
        doc_vectors.append(embeddings)
    
    return doc_vectors

In [ ]:


# Function to compute similarity using cosine similarity
def compute_similarity(doc_vectors):
    cosine_sim = cosine_similarity(doc_vectors)
    return cosine_sim

# Example Documents
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast, brown fox leaps over a lazy dog.",
    "A cat sits on the mat."
]

# Preprocess the documents
preprocessed_documents = [preprocess_text(doc) for doc in documents]

# Vectorization Methods
# 1. TF-IDF
tfidf_matrix = tfidf_vectorizer(preprocessed_documents)
tfidf_similarity = cosine_similarity(tfidf_matrix)

# 2. Word2Vec
word2vec_vectors = word2vec_vectorizer(preprocessed_documents)
word2vec_similarity = compute_similarity(word2vec_vectors)

# 3. BERT
bert_vectors = bert_vectorizer(preprocessed_documents)
bert_similarity = compute_similarity(bert_vectors)

# Display Similarity Results
print("TF-IDF Cosine Similarity:")
print(tfidf_similarity)

print("\nWord2Vec Cosine Similarity:")
print(word2vec_similarity)

print("\nBERT Cosine Similarity:")
print(bert_similarity)

# You can visualize or further analyze these similarity matrices to understand which documents are most similar.


/home/nsl47/anaconda3/envs/bd/lib/python3.11/site-packages/torch/cuda/__init__.py:628: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/nsl47/anaconda3/envs/bd/lib/python3.11/site-packages/torch/cuda/__init__.py:758: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 9010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count
2025-03-07 12:50:34.108077: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the enviro

TF-IDF Cosine Similarity:
[[1.         0.53634991 0.        ]
 [0.53634991 1.         0.        ]
 [0.         0.         1.        ]]

Word2Vec Cosine Similarity:
[[ 1.0000002   0.68917555 -0.02919958]
 [ 0.68917555  1.0000002   0.05782688]
 [-0.02919958  0.05782688  1.        ]]

BERT Cosine Similarity:
[[1.         0.9500985  0.5872829 ]
 [0.9500985  0.9999999  0.56055367]
 [0.5872829  0.56055367 1.0000005 ]]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Example Documents
# documents = [
#     "The stock market crashed yesterday due to political instability.",
#     "The economy suffered as the political unrest caused a downfall in stock market prices."
# ]
documents = [
    "He loves cricket",
    "She loves cricket"
]

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the documents into TF-IDF vectors
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Calculate cosine similarity between the documents
cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

print("TF-IDF Cosine Similarity:")
print(cosine_sim[0][0])


TF-IDF Cosine Similarity:
0.5031026124151314


In [ ]:
from gensim.models import Word2Vec
import numpy as np

# Example Documents
# documents = [
#     "The stock market crashed yesterday due to political instability.",
#     "The economy suffered as the political unrest caused a downfall in stock market prices."
# ]
documents = [
    "He loves cricket",
    "She loves cricket"
]

# Tokenize documents (split each document into words)
tokenized_documents = [doc.lower().split() for doc in documents]

# Train a Word2Vec model
word2vec_model = Word2Vec(tokenized_documents, vector_size=100, window=5, min_count=1, workers=4)

# Function to average Word2Vec vectors of words in a document
def get_document_vector(doc, model):
    word_vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

# Get vectors for each document
doc1_vector = get_document_vector(tokenized_documents[0], word2vec_model)
doc2_vector = get_document_vector(tokenized_documents[1], word2vec_model)

# Calculate cosine similarity between the document vectors
cosine_sim_word2vec = np.dot(doc1_vector, doc2_vector) / (np.linalg.norm(doc1_vector) * np.linalg.norm(doc2_vector))

print("Word2Vec Cosine Similarity:")
print(cosine_sim_word2vec)


Word2Vec Cosine Similarity:
0.9999999


In [11]:
from transformers import BertTokenizer, BertModel
import torch

# Example Documents
# documents = [
#     "The stock market crashed yesterday due to political instability.",
#     "The economy suffered as the political unrest caused a downfall in stock market prices."
# ]
documents = [
    "He loves cricket",
    "She loves cricket"
]

# Initialize BERT Tokenizer and Model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to get BERT embeddings for a document
def get_bert_embedding(doc):
    inputs = tokenizer(doc, return_tensors='pt', padding=True, truncation=True, max_length=512)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

# Get BERT embeddings for each document
doc1_bert_embedding = get_bert_embedding(documents[0])
doc2_bert_embedding = get_bert_embedding(documents[1])

# Calculate cosine similarity between the BERT embeddings
cosine_sim_bert = np.dot(doc1_bert_embedding, doc2_bert_embedding) / (np.linalg.norm(doc1_bert_embedding) * np.linalg.norm(doc2_bert_embedding))

print("BERT Cosine Similarity:")
print(cosine_sim_bert)


BERT Cosine Similarity:
0.95479345


## Summary:

#### TF-IDF focuses on the importance of unique terms in the documents and calculates similarity based on that.

#### Word2Vec captures semantic meaning through word embeddings, so it's better at capturing the relationship between words.

#### BERT understands the entire context of the sentences, giving the highest similarity score due to its ability to process contextual information.